# Notebook 4: Emergent Strategies

The most fascinating aspect of RLMs: nobody programs these strategies. Given the ability to write code and call themselves recursively, models naturally discover 5 key patterns:

1. **Peeking** — Sampling the beginning of context to understand structure
2. **Grepping** — Using regex/keyword search to find relevant sections
3. **Partition + Map** — Chunking context and processing each chunk with sub-calls
4. **Summarization** — Extracting key info from context subsets
5. **Long-input Long-output** — Building answers programmatically in variables

In this notebook, we'll demonstrate each strategy by running the kind of code an RLM would generate inside our sandbox.

## Setup

We reuse the same pattern from Notebook 03: import `rlm_core`, load sample data, and create a `SimulatedClient` so the notebook runs without a live LLM server.

In [ ]:
import sys
sys.path.insert(0, "..")

from rlm_core import LLMClient, Sandbox

# Load the needle-in-haystack document as our primary context
with open("../data/samples/needle_haystack.txt") as f:
    document = f.read()

print(f"Loaded document: {len(document)} characters")
print(f"Preview: {document[:120]}...")

# Simulated client (same pattern as NB03)
class SimulatedClient:
    """Simulates LLM responses for demonstration."""
    def __init__(self):
        self.call_count = 0

    def completion(self, prompt, **kwargs):
        self.call_count += 1
        class Result:
            prompt_tokens = 10
            completion_tokens = 5
        r = Result()
        r.text = 'FINAL("simulated response")'
        return r

client = SimulatedClient()
print("Setup complete.")

## Strategy 1: Peeking

The model's first move is often to look at the **start** of the context to understand what kind of data it's dealing with. This is cheap (only reads a small slice) and tells the model whether it's looking at JSON, plain text, CSV, code, or something else entirely.

Think of it like a human opening a file and glancing at the first few lines before deciding how to process it.

In [ ]:
# What peeking looks like in practice
peeking_code = '''
# Strategy: PEEKING - examine the structure first
first_500 = context[:500]
print(f"Context starts with: {first_500[:200]}...")
print(f"Total length: {len(context)} chars")

# Check if it looks like JSON, plain text, or structured data
if context.strip().startswith('{') or context.strip().startswith('['):
    print("Format: JSON data")
elif '\\t' in context[:200]:
    print("Format: Tab-separated data")
else:
    print("Format: Plain text")
'''
sb = Sandbox(variables={"context": document})
result = sb.execute(peeking_code)
print("=== Peeking Strategy Output ===")
print(result.stdout)

## Strategy 2: Grepping

Once the model knows the structure, it often searches for specific patterns using regex or keyword matching. This is the "needle in a haystack" strategy — skip reading the entire document and jump straight to what matters.

This is especially effective for tasks like "find the secret code" or "extract all email addresses."

In [ ]:
grepping_code = '''
import re
# Strategy: GREPPING - search for specific patterns
matches = re.findall(r'secret|code|hidden|password', context, re.IGNORECASE)
print(f"Found {len(matches)} keyword matches: {matches}")

# More targeted search
code_match = re.search(r'(?:secret|hidden)\s+code\s+is\s+([A-Z0-9-]+)', context)
if code_match:
    print(f"Found code: {code_match.group(1)}")
    FINAL(code_match.group(1))
else:
    print("No direct match, need to search more carefully")
'''
final_result = {"value": None}
def final_fn(ans): final_result["value"] = str(ans)

sb = Sandbox(variables={"context": document, "FINAL": final_fn})
result = sb.execute(grepping_code)
print("=== Grepping Strategy Output ===")
print(result.stdout)
if final_result["value"]:
    print(f"FINAL answer: {final_result['value']}")

## Strategy 3: Partition + Map

When the context is too large to process at once, or the task requires aggregation (counting, summing, filtering), models discover the classic divide-and-conquer approach:

1. **Partition** the data into manageable chunks
2. **Map** a processing function over each chunk
3. **Reduce** the results into a final answer

This mirrors MapReduce and is particularly effective for aggregation tasks over structured data.

In [ ]:
import json
with open("../data/samples/aggregation_items.json") as f:
    items = json.load(f)
agg_context = json.dumps(items)

partition_code = '''
import json
data = json.loads(context)

# Strategy: PARTITION + MAP - split into chunks and process each
chunk_size = 25
chunks = [data[i:i+chunk_size] for i in range(0, len(data), chunk_size)]
print(f"Split {len(data)} items into {len(chunks)} chunks of ~{chunk_size}")

total_red_large = 0
for i, chunk in enumerate(chunks):
    count = sum(1 for item in chunk if item["color"] == "red" and item["size"] == "large")
    total_red_large += count
    print(f"  Chunk {i}: {count} red+large items")

print(f"\\nTotal red AND large: {total_red_large}")
FINAL(str(total_red_large))
'''
final_result = {"value": None}
sb = Sandbox(variables={"context": agg_context, "FINAL": final_fn, "json": __import__("json")})
result = sb.execute(partition_code)
print("=== Partition + Map Strategy Output ===")
print(result.stdout)

## Strategy 4: Summarization

For multi-document or multi-section tasks, models extract key facts from each section independently, then combine them. This is the strategy behind multi-hop question answering:

1. Split the context into logical sections
2. Summarize each section (extract key entities, relationships, facts)
3. Use the summaries to answer the question

In a full RLM, each summarization step would be a recursive `llm_query()` sub-call. Here we simulate it with simple text extraction.

In [ ]:
import os
# Load multihop docs
docs = []
doc_dir = "../data/samples/multihop_docs"
for fname in sorted(os.listdir(doc_dir)):
    with open(os.path.join(doc_dir, fname)) as f:
        docs.append(f.read())

multi_context = "\n\n---\n\n".join(docs)

summarize_code = '''
# Strategy: SUMMARIZATION - extract key facts from each section
sections = context.split("---")
print(f"Found {len(sections)} sections")

for i, section in enumerate(sections):
    section = section.strip()
    if not section:
        continue
    # Extract first sentence as summary
    first_sentence = section.split('.')[0] + '.'
    print(f"  Section {i}: {first_sentence[:80]}")
'''
sb = Sandbox(variables={"context": multi_context})
result = sb.execute(summarize_code)
print("=== Summarization Strategy Output ===")
print(result.stdout)

## Strategy 5: Long-input Long-output

Some tasks require producing a long, structured output (a report, a transformation, a detailed analysis). The model can't fit this into a single `FINAL("...")` call, so it builds the result programmatically in a variable and uses `FINAL_VAR("variable_name")` to return it.

This strategy turns the model into a data pipeline: read structured input, transform it, and produce structured output.

In [ ]:
long_io_code = '''
# Strategy: LONG-INPUT LONG-OUTPUT - build result in a variable
import json
data = json.loads(context)

# Transform: create a summary report (too long for a single FINAL())
report_lines = ["# Inventory Report", ""]

# Group by color
colors = {}
for item in data:
    c = item["color"]
    colors[c] = colors.get(c, 0) + 1

report_lines.append("## Items by Color")
for color, count in sorted(colors.items()):
    report_lines.append(f"- {color}: {count} items")

report_lines.append("")
report_lines.append("## Items by Size")
sizes = {}
for item in data:
    s = item["size"]
    sizes[s] = sizes.get(s, 0) + 1
for size, count in sorted(sizes.items()):
    report_lines.append(f"- {size}: {count} items")

report = "\\n".join(report_lines)
print(report)
FINAL_VAR("report")
'''
final_result = {"value": None}
def final_var_fn(var_name):
    final_result["value"] = str(sb.get_variable(var_name))
sb = Sandbox(variables={"context": agg_context, "FINAL": final_fn, "FINAL_VAR": final_var_fn, "json": __import__("json")})
result = sb.execute(long_io_code)
print("=== Long I/O Strategy Output ===")
print(result.stdout[:500])

## Strategy Classifier

Can we automatically detect which strategy an LLM used? Let's build a simple heuristic classifier that examines the generated code and identifies which patterns are present.

In [ ]:
import re

def classify_strategy(code: str) -> list[str]:
    """Classify which strategies are present in LLM-generated code."""
    strategies = []
    if re.search(r'context\[:\d+\]|\.[:]\d+', code):
        strategies.append("peeking")
    if re.search(r're\.(search|findall|match)', code):
        strategies.append("grepping")
    if re.search(r'chunk|split|partition|\[i:i\+', code):
        strategies.append("partition+map")
    if re.search(r'llm_query|summary|summarize', code):
        strategies.append("summarization")
    if re.search(r'FINAL_VAR|report|output.*=.*\[\]', code):
        strategies.append("long-io")
    return strategies if strategies else ["unknown"]

# Test it on our examples
print("Peeking code:", classify_strategy(peeking_code))
print("Grepping code:", classify_strategy(grepping_code))
print("Partition code:", classify_strategy(partition_code))
print("Long I/O code:", classify_strategy(long_io_code))

## Why Context-as-Variable Enables This

All five strategies depend on one architectural decision: **context is stored as a variable**, not stuffed into the prompt.

If the context were embedded directly in the prompt:
- The model **couldn't slice** it (no peeking)
- The model **couldn't regex** over it (no grepping)
- The model **couldn't chunk** it programmatically (no partition+map)
- The model **couldn't selectively read** sections (no summarization)
- The model **couldn't transform** it into structured output (no long I/O)

By making context a sandbox variable, the model gains full programmatic access to it. This single design choice is what unlocks all five emergent strategies.

## Key Takeaways

1. **RLMs discover 5 strategies without being programmed to** — peeking, grepping, partition+map, summarization, and long I/O emerge naturally from the ability to write and execute code
2. **Peeking** examines structure, **grepping** finds needles, **partition+map** handles aggregation
3. **The model chooses its strategy based on the task** — just like a human programmer would
4. **Context-as-variable is what makes all of this possible** — without it, none of these strategies would work

## What's Next

In Notebook 5, we'll assemble everything into a complete RLM pipeline and run it on all our sample tasks.